# Visualise a generated flow dataset

Set `DATA` to any generated `.npy` (shape `(n_seqs, n_frames, H, W)`, physical vorticity).
Defaults to the **Re=2000** dataset. Select the **Python (venv-ddpm)** kernel, Run All.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.getcwd()
while ROOT != "/" and not os.path.isdir(os.path.join(ROOT, "data_generation")):
    ROOT = os.path.dirname(ROOT)

# --- choose which generated dataset to view ---
DATA = os.path.join(ROOT, "data_generation", "jaxcfd_re2000_1024to256_40seq.npy")
# DATA = os.path.join(ROOT, "data_generation", "jaxcfd_re1000_1024to256_2seq.npy")

d = np.load(DATA, mmap_mode="r")
print("file:", os.path.basename(DATA), "| shape", d.shape)
samp = np.asarray(d[:4, ::16])
print(f"mean={samp.mean():+.4f}  std={samp.std():.4f}  min={samp.min():.2f}  max={samp.max():.2f}  (physical)")

In [ ]:
# --- sample frame grid (sequences x frames) ---
seqs = [0, 13, 26, 39][: d.shape[0]]
frames = np.linspace(0, d.shape[1] - 1, 6, dtype=int)
vmax = float(np.percentile(np.abs(np.asarray(d[0, frames])), 99))
fig, axes = plt.subplots(len(seqs), len(frames), figsize=(2.7 * len(frames), 2.7 * len(seqs)), squeeze=False)
for i, s in enumerate(seqs):
    for j, f in enumerate(frames):
        axes[i, j].imshow(d[s, f], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if i == 0: axes[i, j].set_title(f"frame {f}")
        if j == 0: axes[i, j].set_ylabel(f"seq {s}", fontsize=11)
fig.suptitle(f"{os.path.basename(DATA)} — sample frames (physical vorticity)", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# --- animate one sequence over time ---
from matplotlib import animation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 100  # MB

ANIM_SEQ, STEP = 0, 3   # which sequence; frame subsample (lower = smoother)
w = np.asarray(d[ANIM_SEQ])
vmax = float(np.percentile(np.abs(w), 99))
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(w[0], cmap="RdBu_r", vmin=-vmax, vmax=vmax, animated=True); ax.axis("off")
ttl = ax.set_title("")
def update(f):
    im.set_array(w[f]); ttl.set_text(f"seq {ANIM_SEQ} - frame {f}/{len(w) - 1}"); return [im]
ani = animation.FuncAnimation(fig, update, frames=range(0, len(w), STEP), interval=100, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())